# Corpus Normalization & Gold Set Finalization

**Purpose:** Applies critical typographical normalizations (converting errant backticks to Armenian buths and standardizing punctuation spacing) across the 120K combined corpus. Subsequently ingests the human-annotated Gold 2K spreadsheet, applies the same normalizations, re-parses the sequences through Stanza to ensure token alignment, and exports the definitive evaluation metadata.

**Inputs:**
- Uncleaned 120K corpus (`filtered_120k_combined.json`).
- Human-annotated Gold 2K spreadsheet (`gold_2k_annotation_Main.xlsx`).

**Outputs:**
- Cleaned 120K corpus (`filtered_120k_combined_cleaned.json`).
- Finalized Gold 2K metadata and text (`gold_2k_metadata_final.json`, `gold_2k_sentences_final.txt`).

**Hardware Requirements:** Standard CPU for text processing and Stanza inference on 2,000 sentences.

**Estimated Runtime:** ~3-5 minutes.

## Setup and Configuration

In [ ]:
import os
import re
import json
import functools
from pathlib import Path

import torch
from openpyxl import load_workbook

DATA_DIR = Path("./data")
CORPUS_120K_PATH = DATA_DIR / "filtered_corpus_v2" / "filtered_120k_combined.json"
CLEANED_120K_PATH = DATA_DIR / "filtered_corpus_v2" / "filtered_120k_combined_cleaned.json"

GOLD_DIR = DATA_DIR / "gold_2k"
GOLD_XLSX_PATH = GOLD_DIR / "gold_2k_annotation_Main.xlsx"
GOLD_OUT_META = GOLD_DIR / "gold_2k_metadata_final.json"
GOLD_OUT_SENTS = GOLD_DIR / "gold_2k_sentences_final.txt"

BACKTICK = "\u0060"
ARMENIAN_BUTH = "\u055D"
LOV_SUFFIX = "լով"
AC_SUFFIX = "ած"
ADDITIVE_DEPRELS = {"obj", "iobj", "obl", "advmod", "acl", "ccomp", "nsubj:pass"}

## Punctuation Normalization Rules

In [ ]:
def replace_backticks(text: str) -> str:
    """Converts errant grave accents (U+0060) to the Armenian modifier letter buth (U+055D)."""
    return text.replace(BACKTICK, ARMENIAN_BUTH)

def fix_spacing(text: str) -> str:
    """Standardizes whitespace surrounding Armenian punctuation markers to conform to orthographic rules.

    Args:
        text (str): The raw Armenian text string.

    Returns:
        str: The orthographically normalized text string.
    """
    arm_range = r'\u0531-\u0556\u0561-\u0587'
    letter = f'[A-Za-z{arm_range}]'
    
    text = re.sub(r' +([,՝:։;!?՞՜)»])', r'\1', text)
    text = re.sub(f'([:։!?])({letter})', r'\1 \2', text)
    text = re.sub(f',({letter})', r', \1', text)
    text = re.sub(f'՝({letter})', r'՝ \1', text)
    text = re.sub(r'([«(]) +', r'\1', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def clean_sentence(text: str) -> str:
    """Applies all typographical normalizations to a single string."""
    return fix_spacing(replace_backticks(text))

def clean_corpus_in_memory(corpus_data: list, text_field: str = "text") -> int:
    """Iterates over corpus records and applies normalizations to sequence strings and tokens."""
    modified_count = 0
    for i, item in enumerate(corpus_data):
        if isinstance(item, dict):
            original = item[text_field]
            cleaned = clean_sentence(original)
            item[text_field] = cleaned
            
            if 'tokens' in item and isinstance(item['tokens'], list):
                for tok in item['tokens']:
                    if isinstance(tok, dict):
                        if 'text' in tok: tok['text'] = replace_backticks(tok['text'])
                        if 'lemma' in tok: tok['lemma'] = replace_backticks(tok['lemma'])
        else:
            original = item
            cleaned = clean_sentence(original)
            corpus_data[i] = cleaned
            
        if original != cleaned:
            modified_count += 1
    return modified_count

## 120K Corpus Processing

In [ ]:
if CORPUS_120K_PATH.exists():
    with open(CORPUS_120K_PATH, 'r', encoding='utf-8') as f:
        corpus_120k = json.load(f)

    clean_corpus_in_memory(corpus_120k, text_field="text")

    with open(CLEANED_120K_PATH, 'w', encoding='utf-8') as f:
        json.dump(corpus_120k, f, ensure_ascii=False)
    
    del corpus_120k

## Gold 2K Ingestion and Alignment

In [ ]:
def load_and_clean_xlsx(filepath: Path) -> list:
    """Extracts the human-annotated rows from the XLSX and normalizes punctuation."""
    wb = load_workbook(filepath, read_only=True)
    ws = wb.active
    records = []
    
    for row in ws.iter_rows(min_row=2, values_only=True):
        if len(row) < 9 or row[1] is None:
            continue
            
        orig = str(row[1])
        corr = str(row[2]) if row[2] else orig
        
        records.append({
            "xlsx_id": row[0],
            "text": clean_sentence(orig),
            "corrected": clean_sentence(corr),
            "notes": str(row[3]) if row[3] else "",
            "rule_hint": str(row[4]) if row[4] else "",
            "position_hint": str(row[5]) if row[5] else "",
            "quality_score": row[7],
            "complexity": str(row[8]) if row[8] else "",
        })
        
    wb.close()
    return records

if GOLD_XLSX_PATH.exists():
    gold_records = load_and_clean_xlsx(GOLD_XLSX_PATH)

## Stanza Parsing and Participle Detection

In [ ]:
original_load = torch.load
torch.load = functools.partial(original_load, weights_only=False)
import stanza

nlp = stanza.Pipeline('hy', processors='tokenize,pos,lemma,depparse', verbose=False)

def detect_participles(tokens: list) -> list:
    """Extracts structural metadata for participles using syntactic dependency trees.

    Args:
        tokens (list): Sequence of token dictionaries from Stanza.

    Returns:
        list: Dictionaries detailing participle positions, types, and additive counts.
    """
    participles = []
    for tok in tokens:
        if tok["upos"] != "VERB":
            continue
            
        text = tok["text"]
        if text.endswith(LOV_SUFFIX):
            ptype = "lov"
        elif text.endswith(AC_SUFFIX):
            ptype = "ats"
        else:
            continue
            
        if ptype == "ats" and tok["dep"] == "amod":
            continue
            
        additives = [t for t in tokens if t["head_index"] == tok["token_index"] and t["dep"] in ADDITIVE_DEPRELS]
        
        root_idx = next((t["token_index"] for t in tokens if t["dep"] == "root"), None)
        position = "UNKNOWN"
        
        if root_idx is not None:
            if tok["token_index"] < root_idx:
                position = "PRE"
            elif tok["token_index"] > root_idx:
                position = "POST"
            else:
                position = "INTRA"
                
            has_subj_before = any(t["dep"] in ("nsubj", "nsubj:pass") and t["token_index"] < tok["token_index"] for t in tokens)
            if has_subj_before and tok["token_index"] < root_idx:
                position = "INTRA"
                
        rule_hint = "R6" if ptype == "ats" and tok["dep"] == "xcomp" else None
        
        participles.append({
            "text": text,
            "type": ptype,
            "token_index": tok["token_index"],
            "additive_count": len(additives),
            "additive_texts": [a["text"] for a in additives],
            "additive_deprels": [a["dep"] for a in additives],
            "position": position,
            "rule_hint": rule_hint,
        })
    return participles

if GOLD_XLSX_PATH.exists():
    for rec in gold_records:
        doc = nlp(rec["text"])
        tokens = []
        for sent in doc.sentences:
            for word in sent.words:
                tokens.append({
                    "text": word.text,
                    "lemma": word.lemma,
                    "upos": word.upos,
                    "xpos": word.xpos if word.xpos else "",
                    "dep": word.deprel,
                    "head": sent.words[word.head - 1].text if word.head > 0 else word.text,
                    "head_index": word.head - 1 if word.head > 0 else word.id - 1,
                    "token_index": word.id - 1
                })
        rec["tokens"] = tokens
        rec["participles"] = detect_participles(tokens)

## Final Export

In [ ]:
if GOLD_XLSX_PATH.exists():
    with open(GOLD_OUT_META, 'w', encoding='utf-8') as f:
        json.dump(gold_records, f, ensure_ascii=False)

    with open(GOLD_OUT_SENTS, 'w', encoding='utf-8') as f:
        for rec in gold_records:
            f.write(rec["text"] + '\n')